<a href="https://colab.research.google.com/github/shaestasaleem/flyrank-machine-learning-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaestasaleem/flyrank-machine-learning-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn

import duckdb
import pandas as pd
import numpy as np
import sklearn

from google.colab import userdata
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit

RANDOM_SEED = 42

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN not found. Add it in Colab Secrets "
        "and enable notebook access."
    )

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

safe_token = hf_token.replace("'", "''")

con.execute(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{safe_token}'
);
""")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

MARCH_DAILY = (
    f"{WAREHOUSE}/fact_content_daily_performance/"
    "month=2026-03/data_0.parquet"
)

print("Warehouse connection ready.")
print("Development month: March 2026")
print("June 2026 remains sealed.")
print("Random seed:", RANDOM_SEED)
print("scikit-learn version:", sklearn.__version__)

Warehouse connection ready.
Development month: March 2026
June 2026 remains sealed.
Random seed: 42
scikit-learn version: 1.6.1


## 1. Method choice and why





I chose Logistic Regression as my first model for the Refresh / Content Opportunity Scoring lane.

My target is binary: whether a content item experiences a future click decline. However, the practical decision is a ranking problem because an editor needs to know which content items should be reviewed first. I therefore use the model's predicted probability as the ranking score and evaluate the ranked list using Precision@K.

I chose Logistic Regression because it is simple and interpretable. My Week-4 baseline was a transparent rule-based ranking, so I want to test whether a simple learned model can improve on that baseline before adding more complexity.

I will compare the model and the Week-4 baseline on the same held-out rows and using the same ranking metrics. A more complex model will only be considered if the comparison earns it.

In [10]:
RANDOM_SEED = 42

LANE = "Refresh / Content Opportunity Scoring"
MODEL_NAME = "Logistic Regression"
TARGET = "is_future_click_decline"

PRIMARY_METRIC = "Precision@20"
SECONDARY_METRIC = "Precision@50"

print("Lane:", LANE)
print("Model:", MODEL_NAME)
print("Target:", TARGET)
print("Random seed:", RANDOM_SEED)
print("Primary metric:", PRIMARY_METRIC)
print("Secondary metric:", SECONDARY_METRIC)

Lane: Refresh / Content Opportunity Scoring
Model: Logistic Regression
Target: is_future_click_decline
Random seed: 42
Primary metric: Precision@20
Secondary metric: Precision@50




## 2. Split design

I use a grouped train/test split based on `client_hash_id`.

The Week-4 baseline was evaluated on the full March development frame and did not use a train/test split. For Week 5, I create an honest held-out split and recompute the Week-4 rule on exactly the same test rows used for the model.

I group by client because one client can have many content items. A random row split could place content from the same client in both training and test data, allowing the model to benefit from client-specific patterns.

The grouped split keeps each client entirely in either training or test. `client_hash_id` is used only for splitting and is never used as a model feature.

I continue using March 2026 as the development month and keep June 2026 sealed. All model inputs are available by the March 24 decision date. Future March 25–31 clicks are used only to create the evaluation outcome.
The grouped folds have no client overlap, so the split prevents direct client leakage.

Client sizes are highly uneven. In three folds, one large client makes up most of the held-out rows, while the remaining folds contain several smaller clients. The held-out future-decline rate also varies across folds.

I treat this variation as part of the evaluation rather than trying to hide it. I will report performance for every fold as well as the average, because a single held-out client mix could make the model look better or worse than it really is.
The grouped folds have no client overlap, so the split prevents direct client leakage.

Client sizes are highly uneven. In three folds, one large client makes up most of the held-out rows, while the remaining folds contain several smaller clients. The held-out future-decline rate also varies across folds.

I treat this variation as part of the evaluation rather than trying to hide it. I will report performance for every fold as well as the average, because a single held-out client mix could make the model look better or worse than it really is.

In [11]:
# ============================================================
# Rebuild the Week-4 decision-time frame
# ============================================================

signal_frame = con.sql(f"""
WITH daily AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_sum_position
    FROM read_parquet('{MARCH_DAILY}')
    WHERE gsc_data_available IS TRUE
),

windows AS (
    SELECT
        client_hash_id,
        content_hash_id,

        COUNT(DISTINCT report_date) AS available_days,

        -- Previous 7 days: March 11-17
        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-11'
                                     AND DATE '2026-03-17'
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS previous_impressions_7d,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-11'
                                     AND DATE '2026-03-17'
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS previous_clicks_7d,

        -- Recent 7 days: March 18-24
        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-18'
                                     AND DATE '2026-03-24'
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS recent_impressions_7d,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-18'
                                     AND DATE '2026-03-24'
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS recent_clicks_7d,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-18'
                                     AND DATE '2026-03-24'
                THEN COALESCE(gsc_sum_position, 0)
                ELSE 0
            END
        ) AS recent_sum_position_7d,

        -- Future outcome ONLY: March 25-31
        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-25'
                                     AND DATE '2026-03-31'
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS future_clicks_7d

    FROM daily

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM windows

WHERE available_days = 31
  AND previous_impressions_7d > 0
  AND recent_impressions_7d > 0
""").df()


# ============================================================
# Decision-time features
# ============================================================

signal_frame["recent_ctr"] = (
    signal_frame["recent_clicks_7d"]
    / signal_frame["recent_impressions_7d"]
)

signal_frame["recent_avg_position"] = (
    signal_frame["recent_sum_position_7d"]
    / signal_frame["recent_impressions_7d"]
)

signal_frame["pre_decision_click_change_7d"] = (
    signal_frame["recent_clicks_7d"]
    - signal_frame["previous_clicks_7d"]
)

# Future data is used ONLY as the outcome.
signal_frame["future_click_decline"] = (
    signal_frame["future_clicks_7d"]
    < signal_frame["recent_clicks_7d"]
).astype(int)


# ============================================================
# Same clean Week-4 modeling universe
# ============================================================

model_frame = signal_frame[
    (signal_frame["recent_impressions_7d"] >= 20)
    & (signal_frame["recent_clicks_7d"] > 0)
].copy()


# Position bucket is a decision-time feature.
model_frame["position_bucket"] = pd.cut(
    model_frame["recent_avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "20+"],
    include_lowest=True
)


# ============================================================
# Honest grouped split by client
# ============================================================

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_SEED
)

train_idx, test_idx = next(
    splitter.split(
        model_frame,
        y=model_frame["future_click_decline"],
        groups=model_frame["client_hash_id"]
    )
)

train_df = model_frame.iloc[train_idx].copy()
test_df = model_frame.iloc[test_idx].copy()


# ============================================================
# Learn expected CTR ONLY from training clients
# ============================================================

train_position_ctr = (
    train_df
    .groupby(
        "position_bucket",
        observed=True
    )["recent_ctr"]
    .median()
)

train_df["expected_ctr_for_position"] = (
    train_df["position_bucket"]
    .map(train_position_ctr)
    .astype(float)
)

test_df["expected_ctr_for_position"] = (
    test_df["position_bucket"]
    .map(train_position_ctr)
    .astype(float)
)

train_df["ctr_vs_expected"] = (
    train_df["recent_ctr"]
    / train_df["expected_ctr_for_position"].replace(0, np.nan)
)

test_df["ctr_vs_expected"] = (
    test_df["recent_ctr"]
    / test_df["expected_ctr_for_position"].replace(0, np.nan)
)


# ============================================================
# Split checks
# ============================================================

train_clients = set(train_df["client_hash_id"])
test_clients = set(test_df["client_hash_id"])

overlap = train_clients.intersection(test_clients)

print("Total modeling rows:", f"{len(model_frame):,}")
print("Training rows:", f"{len(train_df):,}")
print("Test rows:", f"{len(test_df):,}")

print()
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(overlap))

print()
print(
    "Training future-decline rate:",
    round(train_df["future_click_decline"].mean(), 4)
)

print(
    "Test future-decline rate:",
    round(test_df["future_click_decline"].mean(), 4)
)

assert len(overlap) == 0, "Client leakage: same client appears in train and test."

assert not train_df["ctr_vs_expected"].isna().any(), (
    "Missing train ctr_vs_expected values."
)

assert not test_df["ctr_vs_expected"].isna().any(), (
    "Missing test ctr_vs_expected values."
)

print("\nPASS: grouped client split has no client overlap.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total modeling rows: 27,186
Training rows: 5,573
Test rows: 21,613

Training clients: 24
Test clients: 7
Client overlap: 0

Training future-decline rate: 0.5091
Test future-decline rate: 0.561

PASS: grouped client split has no client overlap.


In [12]:
from sklearn.model_selection import GroupKFold

N_SPLITS = 5

gkf = GroupKFold(n_splits=N_SPLITS)

fold_summary = []

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(
        model_frame,
        y=model_frame["future_click_decline"],
        groups=model_frame["client_hash_id"]
    ),
    start=1
):
    fold_train = model_frame.iloc[train_idx]
    fold_test = model_frame.iloc[test_idx]

    train_clients = set(fold_train["client_hash_id"])
    test_clients = set(fold_test["client_hash_id"])

    fold_summary.append({
        "fold": fold,
        "train_rows": len(fold_train),
        "test_rows": len(fold_test),
        "train_clients": len(train_clients),
        "test_clients": len(test_clients),
        "client_overlap": len(train_clients & test_clients),
        "train_base_rate": fold_train["future_click_decline"].mean(),
        "test_base_rate": fold_test["future_click_decline"].mean(),
    })

fold_summary = pd.DataFrame(fold_summary)

display(fold_summary.round(4))

,fold,train_rows,test_rows,train_clients,test_clients,client_overlap,train_base_rate,test_base_rate
0,1,20166,7020,30,1,0,0.5495,0.5528
1,2,21491,5695,30,1,0,0.5466,0.5645
2,3,22260,4926,30,1,0,0.5489,0.5568
3,4,22413,4773,17,14,0,0.5432,0.5839
4,5,22414,4772,17,14,0,0.5633,0.4895



## 3. Train + compare vs my baseline

I train a Logistic Regression model using only information available by the March 24 decision date.

I use a small set of interpretable search-performance features: recent impressions, recent clicks, recent CTR, recent average position, recent click change, and CTR relative to the expected level for the search-position bucket.

For every validation fold, the expected CTR for each position bucket is calculated from the training clients only and then applied to the held-out clients.

I also rebuild my exact Week-4 rule inside every fold. The baseline selects content whose recent clicks increased and whose CTR is more than 1.20 times the expected CTR for its position bucket, then ranks the selected items by recent impressions.

The baseline and Logistic Regression are therefore evaluated on exactly the same held-out rows. My main metrics are Precision@20 and Precision@50. I also report ROC-AUC as a broader ranking measure.

I do not expect the model to win automatically. If it does not improve on the simple baseline, I will report that result rather than adding complexity only to obtain a better-looking score.

In [13]:
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

RANDOM_SEED = 42
N_SPLITS = 5

FEATURES = [
    "recent_impressions_7d",
    "recent_clicks_7d",
    "recent_ctr",
    "recent_avg_position",
    "pre_decision_click_change_7d",
    "ctr_vs_expected",
]

TARGET = "future_click_decline"


def precision_at_k(y_true, scores, k):
    """Precision among the k highest-scored rows."""
    if len(y_true) < k:
        return np.nan

    order = np.argsort(np.asarray(scores))[::-1][:k]
    return np.asarray(y_true)[order].mean()


def baseline_precision_at_k(test_part, k):
    """
    Reproduce the Week-4 baseline:
    - recent click change > 0
    - CTR > 1.20x expected CTR
    - rank matching rows by recent impressions
    """
    eligible = test_part[
        (test_part["pre_decision_click_change_7d"] > 0)
        & (test_part["ctr_vs_expected"] > 1.20)
    ].copy()

    if len(eligible) < k:
        return np.nan

    ranked = eligible.sort_values(
        "recent_impressions_7d",
        ascending=False
    )

    return ranked.head(k)[TARGET].mean()


gkf = GroupKFold(n_splits=N_SPLITS)

fold_results = []
oof_parts = []
coefficient_rows = []

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(
        model_frame,
        y=model_frame[TARGET],
        groups=model_frame["client_hash_id"]
    ),
    start=1
):

    train_df = model_frame.iloc[train_idx].copy()
    test_df = model_frame.iloc[test_idx].copy()

    # --------------------------------------------------------
    # Expected CTR learned ONLY from training clients
    # --------------------------------------------------------

    train_position_ctr = (
        train_df
        .groupby(
            "position_bucket",
            observed=True
        )["recent_ctr"]
        .median()
    )

    train_df["expected_ctr_for_position"] = (
        train_df["position_bucket"]
        .map(train_position_ctr)
        .astype(float)
    )

    test_df["expected_ctr_for_position"] = (
        test_df["position_bucket"]
        .map(train_position_ctr)
        .astype(float)
    )

    train_df["ctr_vs_expected"] = (
        train_df["recent_ctr"]
        / train_df["expected_ctr_for_position"].replace(0, np.nan)
    )

    test_df["ctr_vs_expected"] = (
        test_df["recent_ctr"]
        / test_df["expected_ctr_for_position"].replace(0, np.nan)
    )

    # Safety checks
    assert not train_df[FEATURES].isna().any().any()
    assert not test_df[FEATURES].isna().any().any()

    train_clients = set(train_df["client_hash_id"])
    test_clients = set(test_df["client_hash_id"])

    assert len(train_clients & test_clients) == 0

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = Pipeline([
        ("scale", StandardScaler()),
        (
            "logistic",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_SEED
            )
        ),
    ])

    X_train = train_df[FEATURES]
    y_train = train_df[TARGET]

    X_test = test_df[FEATURES]
    y_test = test_df[TARGET]

    model.fit(X_train, y_train)

    model_score = model.predict_proba(X_test)[:, 1]

    # --------------------------------------------------------
    # Exact Week-4 baseline on SAME held-out rows
    # --------------------------------------------------------

    baseline_match = (
        (test_df["pre_decision_click_change_7d"] > 0)
        & (test_df["ctr_vs_expected"] > 1.20)
    )

    baseline_score = np.where(
        baseline_match,
        test_df["recent_impressions_7d"],
        0
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    base_rate = y_test.mean()

    baseline_p20 = baseline_precision_at_k(test_df, 20)
    baseline_p50 = baseline_precision_at_k(test_df, 50)

    model_p20 = precision_at_k(y_test, model_score, 20)
    model_p50 = precision_at_k(y_test, model_score, 50)

    baseline_auc = roc_auc_score(y_test, baseline_score)
    model_auc = roc_auc_score(y_test, model_score)

    fold_results.append({
        "fold": fold,
        "test_rows": len(test_df),
        "test_clients": len(test_clients),
        "base_rate": base_rate,
        "baseline_p20": baseline_p20,
        "model_p20": model_p20,
        "baseline_p50": baseline_p50,
        "model_p50": model_p50,
        "baseline_auc": baseline_auc,
        "model_auc": model_auc,
    })

    # --------------------------------------------------------
    # Save held-out predictions for later error analysis
    # --------------------------------------------------------

    fold_oof = test_df[
        FEATURES + [TARGET]
    ].copy()

    fold_oof["fold"] = fold
    fold_oof["model_score"] = model_score
    fold_oof["baseline_score"] = baseline_score
    fold_oof["baseline_match"] = baseline_match.to_numpy()

    oof_parts.append(fold_oof)

    # --------------------------------------------------------
    # Save standardized Logistic Regression coefficients
    # --------------------------------------------------------

    coefficients = model.named_steps["logistic"].coef_[0]

    for feature, coefficient in zip(FEATURES, coefficients):
        coefficient_rows.append({
            "fold": fold,
            "feature": feature,
            "coefficient": coefficient,
        })


# ------------------------------------------------------------
# Fold-level table
# ------------------------------------------------------------

fold_results = pd.DataFrame(fold_results)

print("Fold-level model vs baseline results")
display(fold_results.round(4))


# ------------------------------------------------------------
# Average comparison table
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline rule",
        "Logistic Regression"
    ],
    "Precision@20": [
        fold_results["baseline_p20"].mean(),
        fold_results["model_p20"].mean(),
    ],
    "Precision@50": [
        fold_results["baseline_p50"].mean(),
        fold_results["model_p50"].mean(),
    ],
    "ROC-AUC": [
        fold_results["baseline_auc"].mean(),
        fold_results["model_auc"].mean(),
    ],
})

print("\nAverage across grouped folds")
display(comparison.round(4))


# ------------------------------------------------------------
# Variability receipt
# ------------------------------------------------------------

variability = pd.DataFrame({
    "metric": [
        "Base rate",
        "Baseline Precision@20",
        "Model Precision@20",
        "Baseline Precision@50",
        "Model Precision@50",
        "Baseline ROC-AUC",
        "Model ROC-AUC",
    ],
    "mean": [
        fold_results["base_rate"].mean(),
        fold_results["baseline_p20"].mean(),
        fold_results["model_p20"].mean(),
        fold_results["baseline_p50"].mean(),
        fold_results["model_p50"].mean(),
        fold_results["baseline_auc"].mean(),
        fold_results["model_auc"].mean(),
    ],
    "std_across_folds": [
        fold_results["base_rate"].std(),
        fold_results["baseline_p20"].std(),
        fold_results["model_p20"].std(),
        fold_results["baseline_p50"].std(),
        fold_results["model_p50"].std(),
        fold_results["baseline_auc"].std(),
        fold_results["model_auc"].std(),
    ],
})

print("\nMean and fold-to-fold variability")
display(variability.round(4))


# Objects used in Section 4
oof_predictions = pd.concat(
    oof_parts,
    ignore_index=True
)

coefficient_table = pd.DataFrame(
    coefficient_rows
)

print("\nOOF rows saved for error analysis:",
      f"{len(oof_predictions):,}")

Fold-level model vs baseline results


,fold,test_rows,test_clients,base_rate,baseline_p20,model_p20,baseline_p50,model_p50,baseline_auc,model_auc
0,1,7020,1,0.5528,0.75,0.75,0.64,0.90,0.5935,0.7078
1,2,5695,1,0.5645,0.75,0.90,0.72,0.92,0.5976,0.7035
2,3,4926,1,0.5568,0.55,0.80,0.70,0.76,0.5742,0.6473
3,4,4773,14,0.5839,0.65,0.85,0.72,0.84,0.6241,0.6975
4,5,4772,14,0.4895,0.65,0.75,0.68,0.78,0.4332,0.5886



Average across grouped folds


,method,Precision@20,Precision@50,ROC-AUC
0,Week-4 baseline rule,0.67,0.692,0.5645
1,Logistic Regression,0.81,0.840,0.6689



Mean and fold-to-fold variability


,metric,mean,std_across_folds
0,Base rate,0.5495,0.0356
1,Baseline Precision@20,0.6700,0.0837
2,Model Precision@20,0.8100,0.0652
3,Baseline Precision@50,0.6920,0.0335
4,Model Precision@50,0.8400,0.0707
5,Baseline ROC-AUC,0.5645,0.0755
6,Model ROC-AUC,0.6689,0.0511



OOF rows saved for error analysis: 27,186


## 4. Errors and interpretation


The model beats my baseline on the grouped March validation, but I do not want to trust the headline metrics without inspecting its behavior.

For feature interpretation, I use the standardized Logistic Regression coefficients from each fold. Because the inputs were standardized before training, the coefficient magnitudes are more comparable than raw-scale coefficients.

For error analysis, I use 0.50 only as a diagnostic probability threshold. This is not my final editorial action threshold because the practical problem is ranking, not ordinary binary classification.

I inspect false positives, false negatives, performance across search-position ranges, and several concrete wrong cases. I do not display client or content identifiers.
### Interpretation

The three strongest features were recent CTR, pre-decision click change, and CTR relative to the expected level for the search-position bucket.

The positive coefficient on recent CTR means higher recent CTR was associated with a higher predicted probability of the future-decline outcome in this March experiment. Recent click change was also positive, which is consistent with my Week-4 signal audit: content that had recently increased was often more likely to decline in the following week, possibly because of short-term mean reversion.

CTR relative to expected position was also important, although its coefficient varied more across folds. I therefore treat it as useful but less stable than a single coefficient might suggest.

Recent impressions had almost no average coefficient, so the Logistic Regression did not appear to depend strongly on exposure alone.

In [14]:
# ============================================================
# SECTION 4 — Errors and interpretation
# ============================================================

# ------------------------------------------------------------
# 1. What features does Logistic Regression lean on?
# ------------------------------------------------------------

feature_summary = (
    coefficient_table
    .groupby("feature")
    .agg(
        mean_coefficient=("coefficient", "mean"),
        mean_abs_coefficient=(
            "coefficient",
            lambda x: np.mean(np.abs(x))
        ),
        std_coefficient=("coefficient", "std"),
    )
    .sort_values(
        "mean_abs_coefficient",
        ascending=False
    )
    .reset_index()
)

print("Feature influence across grouped folds")
display(feature_summary.round(4))

print("\nTop 3 features by mean absolute standardized coefficient")
display(
    feature_summary[
        [
            "feature",
            "mean_coefficient",
            "mean_abs_coefficient",
            "std_coefficient",
        ]
    ].head(3).round(4)
)


# ------------------------------------------------------------
# 2. Diagnostic classification errors
# ------------------------------------------------------------
# 0.50 is used ONLY for error inspection.
# The actual task remains ranking by model probability.

errors = oof_predictions.copy()

errors["predicted_label_050"] = (
    errors["model_score"] >= 0.50
).astype(int)

errors["error_type"] = np.select(
    [
        (errors[TARGET] == 0)
        & (errors["predicted_label_050"] == 1),

        (errors[TARGET] == 1)
        & (errors["predicted_label_050"] == 0),
    ],
    [
        "false_positive",
        "false_negative",
    ],
    default="correct"
)

error_counts = (
    errors["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .reset_index(name="n")
)

error_counts["pct"] = (
    error_counts["n"] / len(errors)
)

print("\nDiagnostic error counts at probability threshold 0.50")
display(error_counts.round(4))


# ------------------------------------------------------------
# 3. Error rate by search-position range
# ------------------------------------------------------------

errors["position_range"] = pd.cut(
    errors["recent_avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "20+"],
    include_lowest=True
)

errors["is_error"] = (
    errors["error_type"] != "correct"
).astype(int)

position_error_table = (
    errors
    .groupby(
        "position_range",
        observed=True
    )
    .agg(
        n=(TARGET, "size"),
        observed_decline_rate=(TARGET, "mean"),
        mean_model_score=("model_score", "mean"),
        diagnostic_error_rate=("is_error", "mean"),
    )
    .reset_index()
)

print("\nErrors by recent average-position range")
display(position_error_table.round(4))


# ------------------------------------------------------------
# 4. Error rate by recent exposure range
# ------------------------------------------------------------

errors["impression_quartile"] = pd.qcut(
    errors["recent_impressions_7d"],
    q=4,
    labels=[
        "Q1_lowest",
        "Q2",
        "Q3",
        "Q4_highest",
    ],
    duplicates="drop"
)

exposure_error_table = (
    errors
    .groupby(
        "impression_quartile",
        observed=True
    )
    .agg(
        n=(TARGET, "size"),
        observed_decline_rate=(TARGET, "mean"),
        mean_model_score=("model_score", "mean"),
        diagnostic_error_rate=("is_error", "mean"),
    )
    .reset_index()
)

print("\nErrors by recent-impression quartile")
display(exposure_error_table.round(4))


# ------------------------------------------------------------
# 5. Concrete wrong cases — no IDs displayed
# ------------------------------------------------------------

case_columns = [
    "fold",
    "model_score",
    TARGET,
    "recent_impressions_7d",
    "recent_clicks_7d",
    "recent_ctr",
    "recent_avg_position",
    "pre_decision_click_change_7d",
    "ctr_vs_expected",
]

false_positives = (
    errors[
        errors["error_type"] == "false_positive"
    ]
    .sort_values(
        "model_score",
        ascending=False
    )
    [case_columns]
    .head(3)
)

false_negatives = (
    errors[
        errors["error_type"] == "false_negative"
    ]
    .sort_values(
        "model_score",
        ascending=True
    )
    [case_columns]
    .head(3)
)

print("\nThree confident false positives")
display(false_positives.round(4))

print("\nThree confident false negatives")
display(false_negatives.round(4))


# ------------------------------------------------------------
# 6. Quick sanity check for suspiciously strong feature effects
# ------------------------------------------------------------

max_mean_abs_coef = (
    feature_summary["mean_abs_coefficient"].max()
)

print(
    "\nLargest mean absolute standardized coefficient:",
    round(max_mean_abs_coef, 4)
)

print(
    "Overall grouped ROC-AUC finding:",
    "moderate rather than near-perfect."
)

print(
    "Interpretation check:",
    "A near-perfect score or one overwhelmingly dominant feature "
    "would require another leakage audit."
)


Feature influence across grouped folds


,feature,mean_coefficient,mean_abs_coefficient,std_coefficient
0,recent_ctr,0.3673,0.3673,0.2672
1,pre_decision_click_change_7d,0.3373,0.3373,0.1080
2,ctr_vs_expected,0.2273,0.3006,0.2598
3,recent_avg_position,0.1335,0.1335,0.0213
4,recent_clicks_7d,-0.1087,0.1087,0.0181
5,recent_impressions_7d,-0.0055,0.0096,0.0130



Top 3 features by mean absolute standardized coefficient


,feature,mean_coefficient,mean_abs_coefficient,std_coefficient
0,recent_ctr,0.3673,0.3673,0.2672
1,pre_decision_click_change_7d,0.3373,0.3373,0.1080
2,ctr_vs_expected,0.2273,0.3006,0.2598



Diagnostic error counts at probability threshold 0.50


...


Three confident false positives


,fold,model_score,future_click_decline,recent_impressions_7d,recent_clicks_7d,recent_ctr,recent_avg_position,pre_decision_click_change_7d,ctr_vs_expected
23436,5,0.9999,0,91.0,9.0,0.0989,3.4505,-2.0,27.8407
21658,4,0.9998,0,64.0,6.0,0.0938,3.6406,1.0,25.1250
21429,4,0.9998,0,42.0,3.0,0.0714,39.9048,-2.0,31.3571



Three confident false negatives


,fold,model_score,future_click_decline,recent_impressions_7d,recent_clicks_7d,recent_ctr,recent_avg_position,pre_decision_click_change_7d,ctr_vs_expected
26188,5,0.0000,1,3805.0,65.0,0.0171,2.6176,-175.0,4.7220
26577,5,0.0000,1,25230.0,248.0,0.0098,2.6789,-131.0,2.7171
25577,5,0.0002,1,946.0,26.0,0.0275,6.4461,-116.0,7.7368



Largest mean absolute standardized coefficient: 0.3673
Overall grouped ROC-AUC finding: moderate rather than near-perfect.
Interpretation check: A near-perfect score or one overwhelmingly dominant feature would require another leakage audit.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.